# CHronoZ — Delta vs Percentage-Change Comparison

Compare the selected transformation used in `notebook/benchmarkplotting.ipynb` against conventional percentage change from `analysisPercentageChange.ipynb`.

For every available selected series, the figure uses a **2×2 layout**: top row = unstandardized transformed changes side by side; bottom row = standardized scores side by side. The same cleaned source series and date index are used. The benchmark side follows the selected transformation family (`log` means `100 × Δln(x)`; `level` means `Δx`), while the percentage-change side is `100 × (xₜ / xₜ₋₁ − 1)`.

**Interpretation caution:** unstandardized values have different units and are not directly comparable in magnitude across methods. Standardized panels make relative extremeness easier to compare, but z-scores do not remove skewness, fat tails, denominator effects, or structural breaks. This notebook is descriptive, not a forecast or causal test.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option('display.max_columns', 30)
candidates = [Path('data/cleaned'), Path('../data/cleaned'), Path('../../data/cleaned')]
cleaned_dir = next((p for p in candidates if p.exists()), None)
if cleaned_dir is None:
    raise FileNotFoundError('Cannot find data/cleaned. Run from repository root, notebook/, or notebook/deltaAnalysis/.')
print('Cleaned data directory:', cleaned_dir.resolve())

## 1. Methodology selection

This is the same 36-series selection and transformation family recorded in `benchmarkplotting.ipynb`. Only `log` and `level` are used to calculate the comparison-side transformed changes; model-family labels are retained for reference. The original notebooks are not modified.

In [ ]:
selection = {
 'HOUST':('log','GOOD'),'PERMIT':('log','GOOD'),'JTSJOL':('log','GOOD'),'OPHNFB':('log','GOOD'),'PPIFID':('log','GOOD'),
 'DRSDCILM':('level','GOOD'),'DRTSCILM':('level','GOOD'),'IMPGSC1':('log','GOOD'),'EXPGSC1':('log','GOOD'),
 'PCEPILFE':('log','GOOD'),'PCEPI':('log','GOOD'),'CPIAUCSL':('log','GOOD'),'CPILFESL':('log','GOOD'),
 'DRBLACBS':('level','GOOD'),'TCU':('level','GOOD'),'CIVPART':('level','GOOD'),'FEDFUNDS':('level','GOOD'),
 'PPIACO':('log','REGIME-SENSITIVE'),'PCEC96':('log','ROBUST-CALIBRATION'),'W875RX1':('log','GOOD'),
 'RRSFS':('log','GOOD'),'CES0500000003':('log','GOOD'),'INDPRO':('log','GOOD'),'IPMAN':('log','GOOD'),
 'TOTALSL':('log','GOOD'),'BUSLOANS':('log','GOOD'),'M2SL':('log','GOOD'),'AWHAETP':('level','DISCRETE'),
 'AWHMAN':('level','GOOD'),'RPI':('log','GOOD'),'PAYEMS':('log','GOOD'),'ICSA':('log','ROBUST'),
 'WALCL':('log','REGIME-ROBUST'),'WRESBAL':('log','REGIME-ROBUST'),'TOTRESNS':('log','STRUCTURAL-ROBUST'),
 'UNRATE':('level','DISCRETE-SHOCK')
}

def read_series(series_id):
    path = cleaned_dir / f'{series_id}.csv'
    if not path.exists():
        return None
    df = pd.read_csv(path)
    if 'value' not in df.columns:
        return None
    value = pd.to_numeric(df['value'], errors='coerce')
    if 'date' in df.columns:
        dates = pd.to_datetime(df['date'], errors='coerce')
    elif 'observation_date' in df.columns:
        dates = pd.to_datetime(df['observation_date'], errors='coerce')
    else:
        dates = pd.Series(pd.RangeIndex(len(df)))
    out = pd.DataFrame({'date': dates, 'value': value}).dropna().sort_values('date')
    out = out.drop_duplicates('date').set_index('date')
    return out['value']

def benchmark_transform(level, transform):
    delta = level.diff()
    dates = level.index
    # Match only plausible consecutive native-frequency observations; skip long data gaps.
    if isinstance(dates, pd.DatetimeIndex) and len(dates) > 1:
        day_gap = dates.to_series().diff().dt.days
        valid_step = day_gap.between(4, 14) | day_gap.between(20, 45) | day_gap.between(70, 120)
        delta = delta.where(valid_step)
    if transform == 'log':
        if (level.dropna() <= 0).any():
            return pd.Series(np.nan, index=level.index, name='benchmark_change')
        change = 100 * (np.log(level) - np.log(level.shift(1)))
        change = change.where(delta.notna())
    else:
        change = delta
    return change.replace([np.inf, -np.inf], np.nan).dropna()

def percentage_transform(level):
    previous = level.shift(1)
    pct = 100 * (level / previous - 1)
    pct = pct.where(previous.ne(0))
    return pct.replace([np.inf, -np.inf], np.nan).dropna()

def zscore(series):
    series = series.dropna()
    sd = series.std(ddof=1)
    if len(series) < 2 or pd.isna(sd) or sd <= 0:
        return pd.Series(dtype=float, index=series.index)
    return (series - series.mean()) / sd


## 2. Side-by-side plots

Each series gets one figure. The top row preserves each method's native transformed units. The bottom row standardizes each method separately using its own sample mean and sample standard deviation. Dates are aligned within each panel using each method's valid observations.

In [ ]:
plot_dir = Path('outputs/benchmark/comparison_plots') if Path('outputs').exists() else Path('../outputs/benchmark/comparison_plots')
plot_dir.mkdir(parents=True, exist_ok=True)
comparison_rows = []
missing_series = []

for sid, (transform, family) in selection.items():
    level = read_series(sid)
    if level is None or level.empty:
        missing_series.append(sid)
        continue
    benchmark_change = benchmark_transform(level, transform).rename('benchmark')
    pct_change = percentage_transform(level).rename('percentage_change')
    benchmark_z = zscore(benchmark_change).rename('benchmark_z')
    pct_z = zscore(pct_change).rename('percentage_z')
    if benchmark_change.empty or pct_change.empty or benchmark_z.empty or pct_z.empty:
        print(f'Skipping {sid}: insufficient data or zero variance in one method')
        continue

    fig, axes = plt.subplots(2, 2, figsize=(16, 9), constrained_layout=True)
    axes[0, 0].plot(benchmark_change.index, benchmark_change.values, linewidth=1)
    axes[0, 0].axhline(0, color='black', linewidth=.8, alpha=.7)
    axes[0, 0].set_title(f'{sid} — benchmark transform ({transform}) | raw')
    axes[0, 0].set_ylabel('100 × Δln(x)' if transform == 'log' else 'Δ level')

    axes[0, 1].plot(pct_change.index, pct_change.values, linewidth=1)
    axes[0, 1].axhline(0, color='black', linewidth=.8, alpha=.7)
    axes[0, 1].set_title(f'{sid} — conventional percentage change | raw')
    axes[0, 1].set_ylabel('Percentage change (%)')

    axes[1, 0].plot(benchmark_z.index, benchmark_z.values, linewidth=1)
    axes[1, 0].axhline(0, color='black', linewidth=.8)
    for k, style in [(1, '--'), (2, ':')]:
        axes[1, 0].axhline(k, linestyle=style, alpha=.65, linewidth=1)
        axes[1, 0].axhline(-k, linestyle=style, alpha=.65, linewidth=1)
    axes[1, 0].set_title('Benchmark transform — standardized')
    axes[1, 0].set_ylabel('Z-score')

    axes[1, 1].plot(pct_z.index, pct_z.values, linewidth=1)
    axes[1, 1].axhline(0, color='black', linewidth=.8)
    for k, style in [(1, '--'), (2, ':')]:
        axes[1, 1].axhline(k, linestyle=style, alpha=.65, linewidth=1)
        axes[1, 1].axhline(-k, linestyle=style, alpha=.65, linewidth=1)
    axes[1, 1].set_title('Percentage change — standardized')
    axes[1, 1].set_ylabel('Z-score')

    for ax in axes.flat:
        ax.set_xlabel('Observation date')
        ax.grid(True, alpha=.2)
    fig.suptitle(f'{sid} | benchmark family: {family}', fontsize=14)
    fig.savefig(plot_dir / f'{sid}_comparison.png', dpi=130)
    plt.show()
    plt.close(fig)

    comparison_rows.append({
        'series': sid, 'benchmark_transform': transform, 'model_family': family,
        'n_benchmark': len(benchmark_change), 'n_percentage_change': len(pct_change),
        'benchmark_mean': benchmark_change.mean(), 'benchmark_std': benchmark_change.std(ddof=1),
        'percentage_mean': pct_change.mean(), 'percentage_std': pct_change.std(ddof=1),
        'benchmark_skew': benchmark_change.skew(), 'percentage_skew': pct_change.skew(),
        'benchmark_excess_kurtosis': benchmark_change.kurt(), 'percentage_excess_kurtosis': pct_change.kurt(),
        'benchmark_max_abs_z': benchmark_z.abs().max(), 'percentage_max_abs_z': pct_z.abs().max(),
        'benchmark_max_abs_z_date': benchmark_z.abs().idxmax(),
        'percentage_max_abs_z_date': pct_z.abs().idxmax(),
    })

comparison_summary = pd.DataFrame(comparison_rows).sort_values('series').reset_index(drop=True)
display(comparison_summary.round(5))
comparison_summary.to_csv(plot_dir / 'comparison_statistics.csv', index=False)
if missing_series:
    print('Selected series missing or unreadable:', ', '.join(missing_series))
print(f'Created {len(comparison_rows)} comparison figures in {plot_dir.resolve()}')

## 3. How to read the comparison

- **Top-left vs top-right:** compares transformed movements in native units. Do not compare vertical magnitudes directly when the transformations use different units.
- **Bottom-left vs bottom-right:** compares how unusual observations are relative to each method's own mean and standard deviation. Large absolute z-scores flag tail observations, not necessarily errors or economic crises.
- **Log benchmark vs percentage change:** for positive values and modest movements, `100 × Δln(x)` and conventional percentage change are often close, but differ for large changes.
- **Level benchmark vs percentage change:** the methods can tell very different stories, particularly for rates, percentages, and series near zero or below zero. Conventional percentage change can be misleading for these series.
- The benchmark side here reproduces the `log`/`level` transformation choice and calendar-gap handling for visualization; it does not apply every special robust calibration rule from `benchmarkplotting.ipynb`. The standardized comparison uses ordinary z-scores for both methods so that this particular comparison is symmetric.

In [ ]:
# Optional compact ranking of series with the most extreme standardized observations.
if not comparison_summary.empty:
    extreme_comparison = comparison_summary[['series', 'benchmark_max_abs_z', 'percentage_max_abs_z', 'benchmark_max_abs_z_date', 'percentage_max_abs_z_date']].copy()
    extreme_comparison['difference_in_max_abs_z'] = extreme_comparison['percentage_max_abs_z'] - extreme_comparison['benchmark_max_abs_z']
    display(extreme_comparison.sort_values('difference_in_max_abs_z', ascending=False).round(4))
